# CADHLLM Phase I — LoRA 訓練（Colab）

> **Base Model**: `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`  
> **訓練方式**: LoRA r=16, alpha=32, 4bit quantization + unsloth  
> **產出**: 標準 PEFT LoRA adapter → `outputs/cadhllm_lora/`

---

**使用方式**：依序執行每個 Cell。確認 Runtime 已選擇 **GPU（T4 或 A100）**。

## 1. 掛載 Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

TRAINING_DIR = "/content/drive/MyDrive/CADHLLM/training"
os.chdir(TRAINING_DIR)
print(f"✅ 工作目錄：{os.getcwd()}")
print(f"   檔案：{os.listdir('.')}")

## 2. 安裝依賴

In [ ]:
!pip install -q unsloth transformers peft trl datasets accelerate bitsandbytes sentencepiece

## 3. 環境檢查

In [ ]:
from trainer import report_environment

info = report_environment()
print("=" * 50)
print(" 環境摘要")
print("=" * 50)
for k, v in info.items():
    print(f"  {k:22s}: {v}")
print("=" * 50)

assert info["cuda_available"], "❌ 請確認 Runtime → Change runtime type → GPU"
print(f"\n✅ GPU 就緒：{info['device_name']}（{info['vram_gb']} GB）")

## 4. 訓練參數

可依需求調整以下參數：

In [ ]:
MAX_STEPS = 300        # 訓練步數（300~500 建議）
DATA_SIZE = 1200       # 合成訓練資料筆數
LORA_R = 16            # LoRA rank
LORA_ALPHA = 32        # LoRA alpha
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
OUTPUT_DIR = "outputs/cadhllm_lora"

## 5. 生成訓練資料

In [ ]:
from data_generator import DataGenerator

gen = DataGenerator()
dataset = gen.generate_synthetic_data(n=DATA_SIZE)
print(f"✅ 生成 {len(dataset)} 筆訓練資料")
print(f"\n--- 第一筆 prompt（前 200 字）---")
print(dataset[0]["prompt"][:200])
print(f"\n--- 第一筆 completion（前 200 字）---")
print(dataset[0]["completion"][:200])

## 6. 開始訓練

預估時間：T4 約 1-2 小時，A100 約 15-20 分鐘。

In [ ]:
from pathlib import Path
from trainer import train

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

result = train(
    data=dataset,
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    learning_rate=LEARNING_RATE,
)

print("\n" + "=" * 50)
print(" 訓練完成")
print("=" * 50)
print(f"  模式 : {result['mode']}")
print(f"  耗時 : {result['elapsed_s']/60:.1f} 分鐘")
print(f"  輸出 : {result['output_dir']}")
print("=" * 50)

## 7. 快速推論測試

用訓練好的 adapter 跑一筆推論，確認輸出格式正確：

In [ ]:
from unsloth import FastLanguageModel
import json

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=OUTPUT_DIR,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

test_prompt = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    "你是 Phase I STEAM 專案規劃師。將使用者的自然語言需求轉換為嚴格的 JSON。\n"
    "規則：只輸出 JSON 物件，不要 Markdown。components 必須包含 Brain/Power/Control。\n"
    "<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>\n\n"
    "我想做一個自動澆水器，用 Arduino 控制，土壤感測器偵測濕度，水泵自動澆水。"
    "<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n\n"
)

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.3, do_sample=True)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("--- 模型輸出 ---")
print(response[:1000])

try:
    parsed = json.loads(response.split("<|eot_id|>")[0] if "<|eot_id|>" in response else response)
    print("\n✅ JSON 解析成功")
    print(f"  project_category: {parsed.get('project_category')}")
    print(f"  components 數量: {len(parsed.get('components', []))}")
except json.JSONDecodeError as e:
    print(f"\n⚠️ JSON 解析失敗: {e}")

## 8. 下載 Adapter

執行後從瀏覽器下載 zip，解壓到主專案 `saved_model/cadhllm_lora/`。

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("cadhllm_lora", "zip", "outputs", "cadhllm_lora")
print(f"✅ 已打包：{zip_path}")
files.download(zip_path)